# 01 - The data: corpora, vectors, and where they live

**What this notebook is for.** Everything downstream (geometry, probes) rests on text corpora
and the emotion vectors extracted from them. This notebook is the catalog: what each dataset
is, who generated it, its quality control, and where the published copy lives.

**Key concepts.**
- *Emotion story corpus*: short texts written to evoke one emotion; the model reads them and
  we average its internal activations per emotion to get one *emotion vector* each.
- *Leakage*: a generated text naming its target emotion. Low leakage means vectors encode the
  concept, not the word.
- *HF*: Hugging Face, where datasets are published (private to the team for now).

**Index.**
1. Corpus catalog
2. Leakage quality control
3. Vector sets and their homes

## 1. Corpus catalog

In [1]:
# this cell tabulates every corpus: size, generator, role
import json
from pathlib import Path

import plotly.graph_objects as go

from emotion_vectors.artifacts import fetch  # local results/ first, HF otherwise

ROOT = Path("..")


def corpus_stats(path):
    rows = [json.loads(l) for l in open(path)]
    return len(rows), sum(len(r["stories"]) for r in rows)


CATALOG = [
    (
        "published stories",
        "snae/emotion_stories_gemma_4_4B",
        "gemma-4-4B (reference authors)",
        "probe extraction, both models",
        171,
        1539,
    ),
    (
        "self stories",
        "results/self_stories_it/dialogues_grouped.jsonl",
        "gemma-4-31b-it (ours)",
        "scale test E6",
        *corpus_stats(fetch("self_stories_it/dialogues_grouped.jsonl")),
    ),
    (
        "dialogues (base)",
        "results/dialogue_stories/dialogues_grouped.jsonl",
        "gemma-4-31b base (ours)",
        "dialogue-transfer E3",
        *corpus_stats(fetch("dialogue_stories/dialogues_grouped.jsonl")),
    ),
    (
        "dialogues (instruct)",
        "results/dialogue_stories_it/dialogues_grouped.jsonl",
        "gemma-4-31b-it (ours)",
        "E5 pilot",
        *corpus_stats(fetch("dialogue_stories_it/dialogues_grouped.jsonl")),
    ),
    (
        "neutral transcripts",
        "results/neutral_transcripts_it/dialogues_grouped.jsonl",
        "gemma-4-31b-it (ours)",
        "confound projection E7",
        *corpus_stats(fetch("neutral_transcripts_it/dialogues_grouped.jsonl")),
    ),
]
fig = go.Figure(
    go.Table(
        header=dict(
            values=["corpus", "location", "generator", "role", "emotions", "texts"], align="left"
        ),
        cells=dict(values=list(zip(*CATALOG)), align="left", height=26),
    )
)
fig.update_layout(
    title="Every corpus in the project, its generator (gemma-4-4B reference, gemma-4-31b, gemma-4-31b-it), and its role",
    height=320,
    margin=dict(t=50, b=10),
)
fig.show()

/Users/abo-tresol/Documents/ai-safety/cbai_project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Leakage quality control

In [2]:
# this cell measures emotion-word leakage per generated corpus and plots the comparison
from emotion_vectors.scoring import leakage

BARS = [
    ("published stories (4B)", 4, 108),
    ("dialogues, base model", *leakage(fetch("dialogue_stories/dialogues_grouped.jsonl"))),
    ("dialogues, instruct", *leakage(fetch("dialogue_stories_it/dialogues_grouped.jsonl"))),
    ("self stories, instruct", *leakage(fetch("self_stories_it/dialogues_grouped.jsonl"))),
]
names = [row[0] for row in BARS]
values = [row[1] / max(row[2], 1) for row in BARS]
fig = go.Figure(
    go.Bar(
        x=names,
        y=[v * 100 for v in values],
        text=[f"{v:.1%}" for v in values],
        textposition="outside",
    )
)
fig.add_annotation(
    text="instruction-tuned generation respects 'do not name the emotion'; the base model does not",
    xref="paper",
    yref="paper",
    x=0.5,
    y=1.13,
    showarrow=False,
)
fig.update_layout(
    title="Emotion-word leakage by corpus: gemma-4-4B, gemma-4-31b base, gemma-4-31b-it (lower is better)",
    yaxis_title="texts naming their emotion (%)",
    height=420,
)
fig.show()

<details><summary><b>How to read this figure</b></summary>

Each bar is one corpus; height is the share of texts that name their target emotion despite instructions not to. The published corpus sets the 4% bar. The base model's 44% is why base-arm dialogue probes carry a lexical confound (handled in the probe notebook).

</details>

## 3. Vector sets and their homes

Each corpus above was run through the extraction pipeline (`src/emotion_vectors/extraction.py`)
to produce per-story activation shards and per-emotion mean vectors at 20 layers. Published
sets, all private team datasets on Hugging Face under `abotresol/`:

| dataset | contents |
|---|---|
| `emotion-vectors-gemma-4-31b` | base-model vectors, published stories |
| `emotion-vectors-gemma-4-31b-it` | instruct-model vectors, published stories |
| `emotion-dialogue-vectors-gemma-4-31b(-it)` | dialogue-derived vectors, both arms |
| `emotion-selfstory-vectors-gemma-4-31b-it` | self-story vectors (scale corpus) |
| `neutral-vectors-gemma-4-31b-it` | neutral-transcript vectors (projection) |
| `emotion-dialogues-...`, `emotion-stories-...`, `neutral-transcripts-...` | the raw corpora |

Reproduce any of them: the corpus row above plus `scripts/extract_emotion_vectors.py`.